# Phase 5 Deployment and API Integration

## Business Goal

Make the trained hospital prediction models accessible to dashboards and internal hospital systems in real time. Phase 5 converts the modeling outputs into a FastAPI service with request validation, response validation, model versioning, and audit-ready prediction logging.

## Deliverable 1: API Source Code

API source file:

```text
api/main.py
```

The FastAPI service exposes the following endpoints:

| Endpoint | Method | Purpose |
| --- | --- | --- |
| `/health` | GET | Confirms API, data schema, model schema, and model artifacts are available. |
| `/schema` | GET | Returns data and model feature schemas. |
| `/summary` | GET | Returns dataset-level summary statistics. |
| `/predict/risk` | POST | Predicts visit risk as `High`, `Low`, or `Medium`. |
| `/predict/claim` | POST | Predicts claim outcome as `Paid`, `Pending`, or `Rejected`. |
| `/score/visit` | POST | Legacy transparent heuristic scoring endpoint retained for Phase 2 compatibility. |

Key implementation details:

- Uses FastAPI for REST API deployment.
- Uses Pydantic models for request and response schema validation.
- Loads saved `.joblib` model pipelines from `model_artifacts/`.
- Returns predicted class and class probabilities.
- Computes a model version from artifact name, modified timestamp, and SHA-256 digest prefix.
- Computes an input feature hash for traceability.
- Appends prediction audit records to `logs/predictions.jsonl`.

GitHub link:

https://github.com/shuuriii/capstone-iitm/blob/main/api/main.py

## Request and Response Validation

The API validates all incoming payloads before scoring. Examples:

- `age` must be between 0 and 120.
- `gender` must be `F` or `M`.
- `visit_month` and `billing_month` must be between 1 and 12.
- `visit_day_of_week` must be between 0 and 6.
- `visit_quarter` must be between 1 and 4.
- Numeric billing and length-of-stay fields must be non-negative.
- Claim model `risk_score` must be `High`, `Low`, or `Medium`.

The response schema returns:

- `prediction`
- `probabilities`
- `model_name`
- `model_version`
- `timestamp_utc`
- `input_feature_hash`
- `audit_log_id`

## Sample Request: Visit Risk Prediction

Endpoint:

```text
POST /predict/risk
```

Sample request:

```json
{
  "age": 67,
  "gender": "F",
  "city": "Mumbai",
  "insurance_provider": "SecureLife",
  "chronic_flag": 1,
  "department": "ICU",
  "visit_type": "ER",
  "doctor_id": 174,
  "length_of_stay_hours_filled": 42.5,
  "visit_frequency": 8,
  "days_since_registration": 310,
  "visit_month": 10,
  "visit_day_of_week": 2,
  "visit_quarter": 4
}
```

Sample response:

```json
{
  "prediction": "Medium",
  "probabilities": {
    "High": 0.322086,
    "Low": 0.313494,
    "Medium": 0.364421
  },
  "model_name": "visit_risk_classifier",
  "model_version": "risk_best_model:20260620064112:3cbf6ff4052f",
  "timestamp_utc": "2026-06-20T06:49:08.401075+00:00",
  "input_feature_hash": "64f1979131292514afd32f058b3f89d41022e6acc11533eae5f8d283829e6f26",
  "audit_log_id": "a66196c6-1d82-461d-97fd-a08cdddcb232"
}
```

## Sample Request: Claim Outcome Prediction

Endpoint:

```text
POST /predict/claim
```

Sample request:

```json
{
  "age": 67,
  "gender": "F",
  "city": "Mumbai",
  "insurance_provider": "SecureLife",
  "chronic_flag": 1,
  "department": "ICU",
  "visit_type": "ER",
  "risk_score": "High",
  "doctor_id": 174,
  "billed_amount": 65000.0,
  "length_of_stay_hours_filled": 42.5,
  "visit_frequency": 8,
  "days_since_registration": 310,
  "visit_month": 10,
  "visit_day_of_week": 2,
  "visit_quarter": 4,
  "billing_month": 10,
  "billing_lag_days": 4
}
```

Sample response:

```json
{
  "prediction": "Paid",
  "probabilities": {
    "Paid": 0.553824,
    "Pending": 0.323724,
    "Rejected": 0.122452
  },
  "model_name": "claim_outcome_classifier",
  "model_version": "claim_best_model:20260620064114:2756c16ae79f",
  "timestamp_utc": "2026-06-20T06:49:08.402222+00:00",
  "input_feature_hash": "35019ba73f2d0ad086f5977fdaeec574c03a88372d12e65bf5c5a9ae81d83be7",
  "audit_log_id": "b5bd5715-892e-46e0-9a8a-b58d28658f7c"
}
```

## Deployment Guide

Deployment guide file:

```text
docs/phase5_deployment_guide.md
```

Local deployment steps:

```bash
python3 -m pip install -r requirements.txt
python3 scripts/build_features.py
python3 scripts/train_phase3_models.py
python3 scripts/evaluate_phase4_models.py
uvicorn api.main:app --host 127.0.0.1 --port 8000
```

Interactive API documentation:

```text
http://127.0.0.1:8000/docs
```

Health check:

```bash
curl http://127.0.0.1:8000/health
```

The API should return `status: ok` and confirm that risk and claim model artifacts are available.

## Audit Logging and Governance

Prediction logs are written to:

```text
logs/predictions.jsonl
```

Each model-backed prediction stores:

- `audit_log_id`
- `timestamp_utc`
- `model_name`
- `model_version`
- `input_feature_hash`
- `prediction`
- `probabilities`

Raw input features are not logged. Instead, the API logs a hash of the feature payload to support traceability while reducing sensitive-data exposure.

Operational recommendations:

- Put the API behind authenticated internal access.
- Rotate or ship prediction logs using an internal logging system.
- Rerun Phase 4 evaluation before promoting new model artifacts.
- Use predictions for prioritization and human review, not autonomous clinical or financial decisions.